In [4]:
import ROOT as r
from itertools import combinations

# opens file and tree
f = r.TFile("/home/masha/actual_data/RunIISummer20UL17NanoAODv9-2_TTtoLNu2Q-1Jets-smeft_MTT-0to700_TuneCP5_13TeV_madgraphMLM-pythia8-2_NANOAODSIM106X_mc2017.root")
tree = f.Get("Events")

nEvents = tree.GetEntries()

# classes definitions

class MyMuon(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
    
    
    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyElectron(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0, cutBased=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
        self.cutBased = cutBased

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyJet(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, btag=0.0, jetid=False):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False

    def IsBTagged(self, threshold):
        return self.btag > threshold

    def HasJetID(self):
        return (self.jetid & 2) != 0

def deltaR(obj1, obj2):
    return obj1.DeltaR(obj2)

# histograms

# muons
h_NMuon   = r.TH1F("h_NMuon", "Number of isolated muons", 7, 0, 7)

# electrons
h_NElectron = r.TH1F("h_NElectron", "Number of isolated electrons", 7, 0, 7)

# jets
h_NJet   = r.TH1F("h_NJet", "Number of jets", 7, 0, 7)
h_NBJet  = r.TH1F("h_NBJet", "Number of b-jets", 7, 0, 7)

h_Jet1_Pt  = r.TH1F("h_Jet1_Pt", "p_{T} of leading jet", 50, 0, 250)
h_Jet2_Pt  = r.TH1F("h_Jet2_Pt", "p_{T} of subleading jet", 50, 0, 250)
h_Jet3_Pt  = r.TH1F("h_Jet3_Pt", "p_{T} of third jet", 50, 0, 250)
h_Jet4_Pt  = r.TH1F("h_Jet4_Pt", "p_{T} of 4th jet", 50, 0, 250)

h_Jet1_Eta = r.TH1F("h_Jet1_Eta", "#eta of leading jet", 50, -4, 4)
h_Jet2_Eta = r.TH1F("h_Jet2_Eta", "#eta of subleading jet", 50, -4, 4)
h_Jet3_Eta = r.TH1F("h_Jet3_Eta", "#eta of third jet", 50, -4, 4)
h_Jet4_Eta = r.TH1F("h_Jet4_Eta", "#eta of 4th jet", 50, -4, 4)

h_BJet1_Pt  = r.TH1F("h_BJet1_Pt", "p_{T} of leading b-jet", 50, 0, 250)
h_BJet2_Pt  = r.TH1F("h_BJet2_Pt", "p_{T} of subleading b-jet", 50, 0, 250)

h_BJet1_Eta = r.TH1F("h_BJet1_Eta", "#eta of leading b-jet", 50, -4, 4)
h_BJet2_Eta = r.TH1F("h_BJet2_Eta", "#eta of subleading b-jet", 50, -4, 4)

# met

h_MET     = r.TH1F("h_MET", "Missing E_{T}", 50, 0, 300)
h_METx    = r.TH1F("h_METx", "MET_x", 50, -300, 300)
h_METy    = r.TH1F("h_METy", "MET_y", 50, -300, 300)
h_METphi  = r.TH1F("h_METphi", "MET #phi", 50, -3.2, 3.2)


# permutations
h_Mlepjet_3j_cases = {}
h_Mhadjet_3j_cases = {}

h_Mlepjet_4pj_cases = {}
h_Mhadtop_4pj_cases = {}

def get_case_hist(hist_dict, name, title):
    if name not in hist_dict:
        hist_dict[name] = r.TH1F(
            name,
            f"{title};Mass [GeV];Events",
            60, 0, 300
        )
        hist_dict[name].Sumw2()

    return hist_dict[name]


# weighted errors
for h in [
    h_NMuon,
    h_NElectron,
    h_NJet, h_NBJet,
    h_Jet1_Pt, h_Jet2_Pt, h_Jet3_Pt, h_Jet4_Pt,
    h_Jet1_Eta, h_Jet2_Eta, h_Jet3_Eta, h_Jet4_Eta,
    h_BJet1_Pt, h_BJet2_Pt,
    h_BJet1_Eta, h_BJet2_Eta,
    h_MET, h_METx, h_METy, h_METphi
    ]:
    h.Sumw2()


# analysis cuts
cuts = {
    "Muon": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "iso_max": 0.15
    },
    "Electron": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "cutBased": 4,
        "iso_max": 0.15
        
    },
    "Jets": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "btag": 0.3040
    },
    "MET": {
        "pt_min": 20.0
    }
}

cuts["Trigger"] = {
    "muon": ["HLT_IsoMu27"],
    "electron": ["HLT_Ele27_WPTight_Gsf", "HLT_Ele32_WPTight_Gsf"]
}

cutflow = {
    "total": 0,

    # triggers
    "pass_mu_trigger": 0,
    "pass_ele_trigger": 0,

    # lepton selection
    "exactly_1_lepton": 0,

    # MET
    "pass_MET": 0,

    # jets
    "3jets": 0,
    "4plus_jets": 0
}

# event loop
for event in range(nEvents):

    cutflow["total"] += 1

    tree.GetEntry(event)
    weight = tree.Generator_weight
    

    #MET cuts
    if not all([
        tree.Flag_goodVertices,
        tree.Flag_globalSuperTightHalo2016Filter,
        tree.Flag_HBHENoiseFilter,
        tree.Flag_HBHENoiseIsoFilter,
        tree.Flag_EcalDeadCellTriggerPrimitiveFilter,
        tree.Flag_BadPFMuonFilter,
        tree.Flag_BadPFMuonDzFilter,
        tree.Flag_eeBadScFilter,
        tree.Flag_ecalBadCalibFilter
        ]):
        continue


    # trigger cuts
    passes_mu_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["muon"]
        )

    passes_ele_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["electron"]
    )   

    if passes_mu_trigger:
        cutflow["pass_mu_trigger"] += 1

    if passes_ele_trigger:
        cutflow["pass_ele_trigger"] += 1

    # muons

    muons = []
    for mu_idx in range(tree.nMuon):
        mu = MyMuon(
            tree.Muon_pt[mu_idx],
            tree.Muon_eta[mu_idx],
            tree.Muon_phi[mu_idx],
            tree.Muon_mass[mu_idx],   
            tree.Muon_miniPFRelIso_all[mu_idx],
            tree.Muon_charge[mu_idx] #unsure
            )
        muons.append(mu)

    iso_muons = [
    m for m in muons
    if m.Pt() > cuts["Muon"]["pt_min"]
    and abs(m.Eta()) < cuts["Muon"]["eta_max"]
    and m.isolation < cuts["Muon"]["iso_max"]
    ]

    iso_muons = sorted(iso_muons, key=lambda m: m.Pt(), reverse=True)
    h_NMuon.Fill(len(iso_muons), weight)

    # electrons
    electrons = []
    for ele_idx in range(tree.nElectron):
        ele = MyElectron(
            tree.Electron_pt[ele_idx],
            tree.Electron_eta[ele_idx],
            tree.Electron_phi[ele_idx],
            tree.Electron_mass[ele_idx],   
            tree.Electron_miniPFRelIso_all[ele_idx],  
            tree.Electron_charge[ele_idx],
            tree.Electron_cutBased[ele_idx]
            )
        electrons.append(ele)

    
    iso_electrons = [
        e for e in electrons
        if e.Pt() > cuts["Electron"]["pt_min"]
        and abs(e.Eta()) < cuts["Electron"]["eta_max"]
        and e.cutBased >= cuts["Electron"]["cutBased"]
        and e.isolation < cuts["Electron"]["iso_max"] 
        ]

    iso_electrons = sorted(iso_electrons, key=lambda e: e.Pt(), reverse=True)

    h_NElectron.Fill(len(iso_electrons), weight)

    # MET from tree
    MET  = tree.MET_pt
    phi  = tree.MET_phi


    METx = MET * r.TMath.Cos(phi)
    METy = MET * r.TMath.Sin(phi)

    h_MET.Fill(MET, weight)
    h_METx.Fill(METx, weight)
    h_METy.Fill(METy, weight)
    h_METphi.Fill(phi, weight)

    n_iso_mu  = len(iso_muons)
    n_iso_ele = len(iso_electrons)

    if n_iso_mu == 1 and n_iso_ele == 0:
        if not passes_mu_trigger:
            continue
        selected_lepton = iso_muons[0]

    elif n_iso_ele == 1 and n_iso_mu == 0:
        if not passes_ele_trigger:
            continue
        selected_lepton = iso_electrons[0]

    else:
        continue
    cutflow["exactly_1_lepton"] += 1
    passes_MET = MET > cuts["MET"]["pt_min"]

    if not passes_MET:
        continue
    cutflow["pass_MET"] += 1

    # jets

    jets = []
    for jet_idx in range(tree.nJet):
        jet = MyJet(
            tree.Jet_pt[jet_idx],
            tree.Jet_eta[jet_idx],
            tree.Jet_phi[jet_idx],
            tree.Jet_mass[jet_idx],    
            tree.Jet_btagDeepFlavB[jet_idx],    
            tree.Jet_jetId[jet_idx] 
            )
        jets.append(jet)

    good_jets = [
    j for j in jets
    if j.Pt() > cuts["Jets"]["pt_min"]
    and abs(j.Eta()) < cuts["Jets"]["eta_max"]
    and j.HasJetID()
    and deltaR(j, selected_lepton) > 0.4
]

    good_jets = sorted(good_jets, key=lambda j: j.Pt(), reverse=True)
    
    # flagging b-tagged jets
    for j in good_jets:
        # j.is_btagged = j.IsBTagged(BTagThreshold)
        j.is_btagged = j.IsBTagged(cuts["Jets"]["btag"])

    bjets = [j for j in good_jets if j.is_btagged]
    bjets = sorted(bjets, key=lambda j: j.Pt(), reverse=True)

    n_jets = len(good_jets)

    
    h_NJet.Fill(len(good_jets), weight)
    h_NBJet.Fill(len(bjets), weight)

    # Leading jets
    if len(good_jets) > 0:
        h_Jet1_Pt.Fill(good_jets[0].Pt(), weight)
        h_Jet1_Eta.Fill(good_jets[0].Eta(), weight)

    if len(good_jets) > 1:
        h_Jet2_Pt.Fill(good_jets[1].Pt(), weight)
        h_Jet2_Eta.Fill(good_jets[1].Eta(), weight)

    if len(good_jets) > 2:
        h_Jet3_Pt.Fill(good_jets[2].Pt(), weight)
        h_Jet3_Eta.Fill(good_jets[2].Eta(), weight)
    if len(good_jets) > 3:
        h_Jet4_Pt.Fill(good_jets[3].Pt(), weight)
        h_Jet4_Eta.Fill(good_jets[3].Eta(), weight)

    # Leading b-jets
    if len(bjets) > 0:
        h_BJet1_Pt.Fill(bjets[0].Pt(), weight)
        h_BJet1_Eta.Fill(bjets[0].Eta(), weight)

    if len(bjets) > 1:
        h_BJet2_Pt.Fill(bjets[1].Pt(), weight)
        h_BJet2_Eta.Fill(bjets[1].Eta(), weight)

    #3 jets
    if n_jets == 3:
        cutflow["3jets"] += 1

        for i_lep in range(3):

            remaining = [i for i in range(3) if i != i_lep]

            label = (
                f"3j_"
                f"lep_j{i_lep+1}_"
                f"had_j{remaining[0]+1}j{remaining[1]+1}"
            )

            lep_jet = good_jets[i_lep]
            had_jet1 = good_jets[remaining[0]]
            had_jet2 = good_jets[remaining[1]]

            lepjet_candidate = selected_lepton + lep_jet
            had_candidate = had_jet1 + had_jet2

            # filling histograms
            get_case_hist(
                h_Mlepjet_3j_cases,
                f"h_Mlepjet_{label}",
                f"Leptonic candidate: {label}"
            ).Fill(lepjet_candidate.M(), weight)

            get_case_hist(
                h_Mhadjet_3j_cases,
                f"h_Mhadjet_{label}",
                f"Hadronic candidate: {label}"
            ).Fill(had_candidate.M(), weight)


    #4-6 jets
    # elif 4 <= n_jets <= 6:
    elif n_jets == 4:
        cutflow["4plus_jets"] += 1

        candidate_jets = good_jets

        # choose jet used with leptonic side
        for i_lep in range(n_jets):

            # choose separate jet used as hadronic-side single jet
            for i_had in range(n_jets):

                if i_had == i_lep:
                    continue

                remaining = [
                    i for i in range(n_jets)
                    if i != i_lep and i != i_had
                ]

                # choose two jets for W
                for i_W1, i_W2 in combinations(remaining, 2):

                    label = (
                        f"{n_jets}j_"
                        f"lep_j{i_lep+1}_"
                        f"had_j{i_had+1}_"
                        f"W_j{i_W1+1}j{i_W2+1}"
                    )

                    lep_jet = candidate_jets[i_lep]
                    had_jet = candidate_jets[i_had]
                    W_jet1 = candidate_jets[i_W1]
                    W_jet2 = candidate_jets[i_W2]

                    W_had = W_jet1 + W_jet2

                    lepjet_candidate = selected_lepton + lep_jet
                    hadtop_candidate = had_jet + W_had

                    get_case_hist(
                        h_Mlepjet_4pj_cases,
                        f"h_Mlepjet_{label}",
                        f"Leptonic candidate: {label}"
                    ).Fill(lepjet_candidate.M(), weight)

                    get_case_hist(
                        h_Mhadtop_4pj_cases,
                        f"h_Mhadtop_{label}",
                        f"Hadronic candidate: {label}"
                    ).Fill(hadtop_candidate.M(), weight)


# histograms

import os

os.makedirs("plots", exist_ok=True)
os.makedirs("plots/3jets", exist_ok=True)
os.makedirs("plots/4plusjets", exist_ok=True)

print("\n=== CUTFLOW ===")
for key, val in cutflow.items():
    print(f"{key:15s}: {val}")


# keep ROOT objects alive
canvases = []
legends = []

# single histogram

def draw_single(hist, canvas_name, title, legend_text, filename):

    c = r.TCanvas(canvas_name, title, 800, 600)

    hist.SetLineWidth(2)
    hist.SetTitle(title)
    hist.Draw("HIST")

    legend = r.TLegend(0.65, 0.78, 0.88, 0.88)
    legend.AddEntry(hist, legend_text, "l")
    legend.Draw()

    c.Update()
    c.SaveAs(filename)

    canvases.append(c)
    legends.append(legend)

# several histograms on one plot

def draw_overlay(hists, labels, colors,
                canvas_name, title, filename):

    c = r.TCanvas(canvas_name, title, 800, 600)

    legend = r.TLegend(0.62, 0.68, 0.88, 0.88)

    max_y = max(h.GetMaximum() for h in hists)

    hists[0].SetMaximum(1.2 * max_y if max_y > 0 else 1)
    hists[0].SetTitle(title)

    for i, (hist, label, color) in enumerate(
        zip(hists, labels, colors)
    ):

        hist.SetLineColor(color)
        hist.SetLineWidth(2)

        if i == 0:
            hist.Draw("HIST")
        else:
            hist.Draw("HIST SAME")

        legend.AddEntry(hist, label, "l")

    legend.Draw()

    c.Update()
    c.SaveAs(filename)

    canvases.append(c)
    legends.append(legend)


# leptons

draw_single(
    h_NMuon,
    "c_muon",
    "Number of isolated muons;Number of muons;Events",
    "Isolated muons",
    "plots/h_muon.png"
)

draw_single(
    h_NElectron,
    "c_electron",
    "Number of isolated electrons;Number of electrons;Events",
    "Isolated electrons",
    "plots/h_electron.png"
)


# jet multiplicities

draw_single(
    h_NJet,
    "c_njet",
    "Number of selected jets;Number of jets;Events",
    "Selected jets",
    "plots/h_njet.png"
)

draw_single(
    h_NBJet,
    "c_nbjet",
    "Number of b-tagged jets;Number of b-tagged jets;Events",
    "b-tagged jets",
    "plots/h_nbjet.png"
)

# jet pT

draw_overlay(
    [h_Jet1_Pt, h_Jet2_Pt, h_Jet3_Pt, h_Jet4_Pt],
    [
        "j1: leading jet",
        "j2: second-leading jet",
        "j3: third-leading jet",
        "j4: fourth-leading jet"
    ],
    [r.kBlack, r.kRed, r.kBlue, r.kGreen + 2],
    "c_jet_pt",
    "Selected jet p_{T};p_{T} [GeV];Events",
    "plots/h_jet_pt.png"
)


# jet eta

draw_overlay(
    [h_Jet1_Eta, h_Jet2_Eta, h_Jet3_Eta, h_Jet4_Eta],
    [
        "j1: leading jet",
        "j2: second-leading jet",
        "j3: third-leading jet",
        "j4: fourth-leading jet"
    ],
    [r.kBlack, r.kRed, r.kBlue, r.kGreen + 2],
    "c_jet_eta",
    "Selected jet #eta;#eta;Events",
    "plots/h_jet_eta.png"
)


# b-tagged jet pT

draw_overlay(
    [h_BJet1_Pt, h_BJet2_Pt],
    [
        "Leading b-tagged jet",
        "Second-leading b-tagged jet"
    ],
    [r.kRed, r.kBlue],
    "c_bjet_pt",
    "b-tagged jet p_{T};p_{T} [GeV];Events",
    "plots/h_bjet_pt.png"
)


# b-tagged jet eta

draw_overlay(
    [h_BJet1_Eta, h_BJet2_Eta],
    [
        "Leading b-tagged jet",
        "Second-leading b-tagged jet"
    ],
    [r.kRed, r.kBlue],
    "c_bjet_eta",
    "b-tagged jet #eta;#eta;Events",
    "plots/h_bjet_eta.png"
)


# MET

draw_single(
    h_MET,
    "c_met",
    "Missing transverse momentum;MET [GeV];Events",
    "MET",
    "plots/h_met.png"
)

draw_single(
    h_METx,
    "c_metx",
    "MET x-component;MET_{x} [GeV];Events",
    "MET x-component",
    "plots/h_metx.png"
)

draw_single(
    h_METy,
    "c_mety",
    "MET y-component;MET_{y} [GeV];Events",
    "MET y-component",
    "plots/h_mety.png"
)

draw_single(
    h_METphi,
    "c_metphi",
    "MET #phi;#phi;Events",
    "MET #phi",
    "plots/h_metphi.png"
)

# reconstruction plots

def draw_case(label, h_lep, h_had, folder):

    c = r.TCanvas(
        f"c_{label}",
        label,
        900, 700
    )

    h_lep.SetLineColor(r.kBlue)
    h_had.SetLineColor(r.kRed)

    h_lep.SetLineWidth(2)
    h_had.SetLineWidth(2)

    max_y = max(
        h_lep.GetMaximum(),
        h_had.GetMaximum()
    )

    h_lep.SetMaximum(
        1.25 * max_y if max_y > 0 else 1
    )

    h_lep.SetTitle(
        f"{label};Reconstructed mass [GeV];Events"
    )

    h_lep.Draw("HIST")
    h_had.Draw("HIST SAME")

    legend = r.TLegend(
        0.45, 0.68,
        0.89, 0.89
    )

    legend.AddEntry(
        h_lep,
        "Blue: lepton + assigned jet",
        "l"
    )

    legend.AddEntry(
        h_had,
        "Red: hadronic jet combination",
        "l"
    )

    legend.AddEntry(
        0,
        "j1, j2, ... ordered by decreasing p_{T}",
        ""
    )

    legend.Draw()

    c.Update()
    c.SaveAs(f"{folder}/{label}.png")

    canvases.append(c)
    legends.append(legend)


# 3-jet cases

for lep_name, h_lep in h_Mlepjet_3j_cases.items():

    label = lep_name.replace(
        "h_Mlepjet_",
        ""
    )

    had_name = f"h_Mhadjet_{label}"

    h_had = h_Mhadjet_3j_cases[had_name]

    draw_case(
        label,
        h_lep,
        h_had,
        "plots/3jets"
    )


# 4-6 jet cases

for lep_name, h_lep in h_Mlepjet_4pj_cases.items():

    label = lep_name.replace(
        "h_Mlepjet_",
        ""
    )

    had_name = f"h_Mhadtop_{label}"

    h_had = h_Mhadtop_4pj_cases[had_name]

    draw_case(
        label,
        h_lep,
        h_had,
        "plots/4plusjets"
    )


# keep windows visible when running as a .py script
for c in canvases:
    c.Update()


=== CUTFLOW ===
total          : 197164
pass_mu_trigger: 38934
pass_ele_trigger: 33116
exactly_1_lepton: 60383
pass_MET       : 55203
3jets          : 15776
4plus_jets     : 17007


Info in <TCanvas::Print>: png file plots/h_muon.png has been created
Info in <TCanvas::Print>: png file plots/h_electron.png has been created
Info in <TCanvas::Print>: png file plots/h_njet.png has been created
Info in <TCanvas::Print>: png file plots/h_nbjet.png has been created
Info in <TCanvas::Print>: png file plots/h_jet_pt.png has been created
Info in <TCanvas::Print>: png file plots/h_jet_eta.png has been created
Info in <TCanvas::Print>: png file plots/h_bjet_pt.png has been created
Info in <TCanvas::Print>: png file plots/h_bjet_eta.png has been created
Info in <TCanvas::Print>: png file plots/h_met.png has been created
Info in <TCanvas::Print>: png file plots/h_metx.png has been created
Info in <TCanvas::Print>: png file plots/h_mety.png has been created
Info in <TCanvas::Print>: png file plots/h_metphi.png has been created
Info in <TCanvas::Print>: png file plots/3jets/3j_lep_j1_had_j2j3.png has been created
Info in <TCanvas::Print>: png file plots/3jets/3j_lep_j2_had_j1j3.p